# Lab 3: Evaluating an AI tool

**AI2Practice Research Lab** | about 2 hours

**The question.** A pneumothorax detector claims 94% sensitivity. What does it get on local data, and is it equal at both sites?

By the end you will be able to build a confusion matrix, compute sensitivity, specificity, PPV and NPV, see how the threshold trades one against another on an ROC curve, and check whether a tool works equally well in two subgroups. You need Labs 1 and 2.

This lab has 4 parts: Part 1, Confusion matrix; Part 2, Sensitivity, specificity, PPV, NPV; Part 3, Thresholds and ROC curves; Part 4, Subgroups.

**The scenario.** Your hospital is considering an AI tool that reads chest X-rays and flags a **pneumothorax** (air between the lung and the chest wall, which can collapse the lung). The vendor's brochure says 94% sensitivity. Before buying, the department ran the tool on recent X-rays from its two sites, A (the main hospital) and B (a smaller affiliated hospital), and radiologists recorded the true answer for each case. This kind of local check is called a **validation study**.

#### How the tasks work

Each task is tagged with a **mode** that tells you what kind of work it is:

| Mode | What you do |
|---|---|
| **Write** | Write a short piece of code in an empty cell. Try it on your own first; a hint and a hidden solution are there if you need them. |
| **Fill in** | Some code is written for you. Replace each `____` with the missing piece. |
| **Reuse** | Import a function you wrote in an earlier lab from your own `toolkit.py`. |
| **Direct the AI** | Ask an AI assistant to write the code, paste it in, run the test, and fill in a short AI-use log. |
| **Fix** | Code "written by an AI" has one bug. Find it and fix it. |
| **Read** | Read some code and predict what it does before you run it. |

Every task ends with a **test cell**. Run it after your code. If it prints `Check passed`, move on. If it shows an `AssertionError`, the message under it tells you what to look at; change your code and run both cells again.

## Set your GitHub username

Your practice data is generated from your GitHub username, so everyone gets a slightly different dataset and the lab page can check your own numbers. Type your username between the quotes (the part after `github.com/` in your profile address), then run the cell with **Shift + Enter**.

In [ ]:
GITHUB_USERNAME = "your-username"   # <- change this to your GitHub username

## Setup

The next cell gets everything ready and imports two functions from **scikit-learn**, the standard Python machine learning library. Run it once each time you open the notebook. It downloads two small helper files from the course repository if they are not here yet: `rl_data.py` makes your practice data, and `toolkit.py` is your own function library (more on that later). Lines starting with `!` are commands for the computer Colab runs on, not Python.

In [ ]:
# Setup: get the helper files (only if they are missing), then make your data
import os
COURSE = "https://raw.githubusercontent.com/UM-IHC-CA2i/CA2i-Research-Launchpad/main/lesson-notebooks"
FORK = f"https://raw.githubusercontent.com/{GITHUB_USERNAME}/CA2i-Research-Launchpad/main/lesson-notebooks"
if not os.path.exists("rl_data.py"):
    !wget -q {COURSE}/rl_data.py
if not os.path.exists("toolkit.py"):
    # Your fork first, so functions you saved in earlier labs come back; else the course copy.
    !wget -q -O toolkit.py {FORK}/toolkit.py || wget -q -O toolkit.py {COURSE}/toolkit.py

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import rl_data
import toolkit
from toolkit import save_to_toolkit

if GITHUB_USERNAME.strip().lower() in ("", "your-username"):
    print("Reminder: put your own GitHub username in the cell above, then run both cells again.")
rl_data.save(3, GITHUB_USERNAME, "detector.csv")   # the AI tool's output on your cases
from sklearn.metrics import roc_curve, roc_auc_score
print("Ready. Data made for GitHub user:", rl_data.normalize_username(GITHUB_USERNAME))

---
## Part 1: Confusion matrix

Your file has one row per X-ray:

- `case_id`: a code for the case
- `site`: A or B
- `truth`: 1 if the radiologists found a pneumothorax, 0 if not. This is the **reference standard** (sometimes called ground truth): the best available answer, which we compare the AI against.
- `score`: the AI's output, a number from 0 to 1. Higher means "more likely pneumothorax". It is not a probability you can take at face value; it is just a ranking.

To turn a score into a yes/no answer we pick a **threshold**: the tool calls a case positive when `score >= threshold`. We start with 0.5.

In [ ]:
det = pd.read_csv("detector.csv")
print("Cases:", len(det), " with pneumothorax:", det["truth"].sum())
det.head()

Comparing the AI's answer with the truth gives four kinds of cases, usually laid out as a 2 x 2 **confusion matrix**:

| | Truth: pneumothorax | Truth: no pneumothorax |
|---|---|---|
| **AI says positive** | True positive (TP) | False positive (FP) |
| **AI says negative** | False negative (FN) | True negative (TN) |

A false negative is a missed pneumothorax; a false positive is a false alarm that someone has to check.

**Mode: Read**

#### Task 1. Which box?

A case has `truth = 0` and `score = 0.62`. With the threshold at 0.5, which box of the confusion matrix does it go in? Set `prediction` to `"TP"`, `"FP"`, `"FN"` or `"TN"`.

In [ ]:
prediction = "____"

<details><summary>Show the answer</summary>

0.62 is at least 0.5, so the AI calls it positive. The truth is 0, so there is no pneumothorax: a false positive (FP).

```python
prediction = "FP"
```

</details>

In [ ]:
assert prediction.strip().upper() == "FP", "0.62 >= 0.5, so the AI says positive; the truth is 0 (no pneumothorax). A positive call without disease is a false positive."
print("Check passed")

**Mode: Fill in**

#### Task 2. Turn scores into answers

Make a new column `pred` that is 1 when the score is at least 0.5 and 0 otherwise. `.astype(int)` turns True/False into 1/0.

In [ ]:
det["pred"] = (det["score"] >= ____).astype(int)
det.head()

<details><summary>Show a solution</summary>

```python
det["pred"] = (det["score"] >= 0.5).astype(int)
det.head()
```

</details>

In [ ]:
assert set(det["pred"]) <= {0, 1}
assert rl_data.check(3, GITHUB_USERNAME, "tp", ((det["pred"] == 1) & (det["truth"] == 1)).sum()), "Use the threshold 0.5 with >=."
print("Check passed")

**Mode: Write**

#### Task 3. Count the four boxes

Count `tp`, `fp`, `fn` and `tn`. Combine two conditions with `&` (and), each in brackets.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* True positives are rows where pred is 1 and truth is 1: `((det["pred"] == 1) & (det["truth"] == 1)).sum()`. `.sum()` counts the True values. Do the same for the other three.

In [ ]:
# Your code here


print("TP", tp, " FP", fp, " FN", fn, " TN", tn)

<details><summary>Show a solution</summary>

```python
tp = ((det["pred"] == 1) & (det["truth"] == 1)).sum()
fp = ((det["pred"] == 1) & (det["truth"] == 0)).sum()
fn = ((det["pred"] == 0) & (det["truth"] == 1)).sum()
tn = ((det["pred"] == 0) & (det["truth"] == 0)).sum()

print("TP", tp, " FP", fp, " FN", fn, " TN", tn)
```

</details>

In [ ]:
assert tp + fp + fn + tn == len(det), "The four counts should add up to all cases."
for name, value in [("tp", tp), ("fp", fp), ("fn", fn), ("tn", tn)]:
    assert rl_data.check(3, GITHUB_USERNAME, name, value), f"{name} is off."
print("Check passed")

**Mode: Reuse**

#### Task 4. Know your cases with count_by() and describe_by()

Before judging a tool, describe the cases it was tested on. Use your toolkit from Labs 1 and 2: `count_by()` for the number of cases per site, and `describe_by()` for the AI score in cases with and without pneumothorax. If the import fails with "still a stub", your saved `toolkit.py` did not reach your fork. Open your Lab 2 notebook, copy your `describe_by()` cell here, run it, and carry on (then save it with `save_to_toolkit(describe_by)` and upload `toolkit.py` as in the Save to GitHub steps).

In [ ]:
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import count_by, describe_by

per_site = ____
score_by_truth = ____
print(per_site, "\n")
score_by_truth

<details><summary>Show a solution</summary>

```python
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import count_by, describe_by

per_site = count_by(det, "site")
score_by_truth = describe_by(det, "truth", "score")
print(per_site, "\n")
score_by_truth
```

</details>

In [ ]:
assert per_site.sum() == len(det) and set(per_site.index) == {"A", "B"}, "per_site should count cases at A and B."
assert score_by_truth.loc[1, "mean"] > score_by_truth.loc[0, "mean"], "Use describe_by(det, 'truth', 'score')."
print("Check passed")

---
## Part 2: Sensitivity, specificity, PPV, NPV

Four numbers summarize the confusion matrix. Each answers a different clinical question:

- **Sensitivity** = TP / (TP + FN). Of the patients who have a pneumothorax, what share does the tool catch? This is the vendor's 94%.
- **Specificity** = TN / (TN + FP). Of the patients without one, what share does it correctly leave alone?
- **Positive predictive value (PPV)** = TP / (TP + FP). When the tool flags a case, how often is it real?
- **Negative predictive value (NPV)** = TN / (TN + FN). When it says negative, how often is that right?

Sensitivity and specificity describe the tool. PPV and NPV also depend on how common the disease is (the **prevalence**), so they change from one hospital to another even with the same tool.

**Mode: Write**

#### Task 5. A toolkit function: sens_spec()

Write `sens_spec(truth, pred)`. Both inputs are lists or columns of 0s and 1s. It should return a dictionary with the keys `tp`, `fp`, `fn`, `tn`, `sensitivity`, `specificity`, `ppv` and `npv`, with the four rates as fractions between 0 and 1. If a denominator is 0, return `float("nan")` for that rate. You will reuse this function in Labs 4 and 5, so make it work for plain lists too.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* Turn both inputs into lists with `list()`, then count with a loop over `zip(truth, pred)`, which walks through both lists side by side. A small helper `rate(a, b)` that returns `a / b`, or `float("nan")` when b is 0, keeps the rest short. Return a dictionary: `{"tp": tp, ...}`.

In [ ]:
# Your code here


m = sens_spec(det["truth"], det["pred"])
m

<details><summary>Show a solution</summary>

```python
def sens_spec(truth, pred):
    """Confusion matrix counts and sensitivity, specificity, PPV, NPV."""
    truth, pred = list(truth), list(pred)
    tp = sum(1 for t, p in zip(truth, pred) if t == 1 and p == 1)
    fp = sum(1 for t, p in zip(truth, pred) if t == 0 and p == 1)
    fn = sum(1 for t, p in zip(truth, pred) if t == 1 and p == 0)
    tn = sum(1 for t, p in zip(truth, pred) if t == 0 and p == 0)

    def rate(a, b):
        return a / b if b else float("nan")

    return {"tp": tp, "fp": fp, "fn": fn, "tn": tn,
            "sensitivity": rate(tp, tp + fn), "specificity": rate(tn, tn + fp),
            "ppv": rate(tp, tp + fp), "npv": rate(tn, tn + fn)}


m = sens_spec(det["truth"], det["pred"])
m
```

</details>

In [ ]:
small = sens_spec([1, 1, 0, 0, 1], [1, 0, 0, 1, 1])
assert (small["tp"], small["fp"], small["fn"], small["tn"]) == (2, 1, 1, 1), "Check your counts on the small example."
assert abs(small["sensitivity"] - 2 / 3) < 1e-9 and abs(small["specificity"] - 0.5) < 1e-9
assert abs(small["ppv"] - 2 / 3) < 1e-9 and abs(small["npv"] - 0.5) < 1e-9
assert sens_spec([0, 0], [0, 0])["sensitivity"] != sens_spec([0, 0], [0, 0])["sensitivity"], "With no positives, sensitivity should be float('nan')."
for key in ["sensitivity", "specificity", "ppv", "npv"]:
    assert rl_data.check(3, GITHUB_USERNAME, key, m[key]), f"{key} on your data is off."
print("Check passed")
save_to_toolkit(sens_spec)

**Mode: Fix**

#### Task 6. Check the AI's summary

A colleague asked an AI assistant for a summary of the four rates as percentages. The code below runs without error, but one of the four rates is calculated wrongly. Run it, compare with your `m` above, find the bug and fix it in place.

In [ ]:
# Written by an AI assistant
def summarize(tp, fp, fn, tn):
    """Sensitivity, specificity, PPV and NPV as percentages (1 decimal)."""
    return {
        "sensitivity": round(100 * tp / (tp + fn), 1),
        "specificity": round(100 * tn / (tn + fn), 1),
        "ppv": round(100 * tp / (tp + fp), 1),
        "npv": round(100 * tn / (tn + fn), 1),
    }

summary = summarize(tp, fp, fn, tn)
summary

<details><summary>Show the bug and the fix</summary>

The specificity line divides by `tn + fn` (everyone the tool called negative). That is the NPV denominator. Specificity is about everyone without disease, so it must divide by `tn + fp`. The wrong denominator gives a plausible-looking number, which is exactly why it is easy to miss.

```python
# Written by an AI assistant (fixed)
def summarize(tp, fp, fn, tn):
    """Sensitivity, specificity, PPV and NPV as percentages (1 decimal)."""
    return {
        "sensitivity": round(100 * tp / (tp + fn), 1),
        "specificity": round(100 * tn / (tn + fp), 1),
        "ppv": round(100 * tp / (tp + fp), 1),
        "npv": round(100 * tn / (tn + fn), 1),
    }

summary = summarize(tp, fp, fn, tn)
summary
```

</details>

In [ ]:
assert summarize(10, 5, 2, 30) == {"sensitivity": 83.3, "specificity": 85.7, "ppv": 66.7, "npv": 93.8}, "summarize() gives a wrong rate on a small example (TP 10, FP 5, FN 2, TN 30)."
for key in ["sensitivity", "specificity", "ppv", "npv"]:
    assert abs(summary[key] - 100 * m[key]) < 0.06, f"summary['{key}'] does not match your sens_spec()."
print("Check passed")

---
## Part 3: Thresholds and ROC curves

The threshold is a choice, not a property of the tool. Lower it and the tool flags more cases: sensitivity rises, specificity falls. Raise it and the opposite happens. A screening tool where a miss is dangerous might use a low threshold; a tool that would trigger invasive follow-up might use a higher one.

A **receiver operating characteristic (ROC) curve** shows every threshold at once: for each one it plots sensitivity (the true positive rate) against 1 minus specificity (the false positive rate). The **area under the curve (AUC)** sums it up in one number: 1.0 is perfect, 0.5 is no better than a coin toss. AUC does not depend on the threshold, which is why it appears in almost every AI paper.

**Mode: Direct the AI**

#### Task 7. A threshold sweep

Ask an AI assistant to write this for you (Colab's built-in Gemini is fine: click **Gemini** at the top right or the sparkle icon in a code cell; any assistant works). Paste the code into the cell below, run it, then run the test. Read the code before you trust it.

Suggested prompt for your AI assistant (adapt it as you like):

> I have a pandas DataFrame `det` with a column `truth` (0 or 1) and a column `score` (0 to 1).
> I also have a function sens_spec(truth, pred) that returns a dict with keys "sensitivity" and "specificity" as fractions.
> Write Python code that, for thresholds 0.05, 0.10, ..., 0.95 (rounded to 2 decimals), predicts 1 when score >= threshold,
> calls sens_spec, and stores the results in a DataFrame called `sweep` with columns threshold, sensitivity, specificity.

In [ ]:
# Paste the AI's code here

<details><summary>Show a reference solution (only after trying with the AI)</summary>

```python
thresholds = np.round(np.arange(0.05, 0.96, 0.05), 2)
rows = []
for t in thresholds:
    r = sens_spec(det["truth"], (det["score"] >= t).astype(int))
    rows.append({"threshold": t, "sensitivity": r["sensitivity"], "specificity": r["specificity"]})
sweep = pd.DataFrame(rows)
sweep
```

</details>

In [ ]:
assert list(sweep.columns)[:3] == ["threshold", "sensitivity", "specificity"], "sweep needs the columns threshold, sensitivity, specificity."
assert len(sweep) == 19, "There should be 19 thresholds, 0.05 to 0.95."
assert abs(sweep["threshold"].iloc[0] - 0.05) < 1e-9 and abs(sweep["threshold"].iloc[-1] - 0.95) < 1e-9
row = sweep.iloc[(sweep["threshold"] - 0.5).abs().argmin()]
assert abs(row["sensitivity"] - m["sensitivity"]) < 0.01, "At threshold 0.5 the sensitivity should match sens_spec(). Are rates fractions (0-1), not percentages?"
assert sweep["sensitivity"].is_monotonic_decreasing, "Sensitivity should only go down as the threshold rises."
print("Check passed")

**AI-use log (Task 7).** Double-click this cell to edit it.

- Tool I used:
- My prompt (paste it):
- What I changed in the AI's code, and why:
- Did the test cell pass on the first try? If not, what was wrong?

**Mode: Fill in**

#### Task 8. Draw the ROC curve

scikit-learn computes the whole curve for you: `roc_curve(truth, score)` returns the false positive rates, true positive rates and thresholds, and `roc_auc_score(truth, score)` the AUC. Note that both take the raw **score**, not the 0/1 prediction. Fill in the blanks.

In [ ]:
fpr, tpr, thr = roc_curve(det["truth"], det[____])
auc = roc_auc_score(det["truth"], ____)

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color="#0D9488", label=f"AI tool (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Coin toss (AUC = 0.5)")
ax.set_xlabel("False positive rate (1 - specificity)")
ax.set_ylabel("True positive rate (sensitivity)")
ax.set_title("ROC curve, pneumothorax detector")
ax.legend()
plt.show()

<details><summary>Show a solution</summary>

```python
fpr, tpr, thr = roc_curve(det["truth"], det["score"])
auc = roc_auc_score(det["truth"], det["score"])

fig, ax = plt.subplots(figsize=(5, 5))
ax.plot(fpr, tpr, color="#0D9488", label=f"AI tool (AUC = {auc:.3f})")
ax.plot([0, 1], [0, 1], "--", color="gray", label="Coin toss (AUC = 0.5)")
ax.set_xlabel("False positive rate (1 - specificity)")
ax.set_ylabel("True positive rate (sensitivity)")
ax.set_title("ROC curve, pneumothorax detector")
ax.legend()
plt.show()
```

</details>

In [ ]:
assert 0.5 < auc <= 1, "AUC should be between 0.5 and 1 for a useful tool. Did you use the score column?"
assert abs(auc - roc_auc_score(det["truth"], det["score"])) < 1e-12
print(f"AUC = {auc:.3f}")
print("Check passed")

**Mode: Write**

#### Task 9. What would it take to reach 94%?

Using `sweep`, find the highest threshold at which sensitivity is still at least 0.94. Store it in `t94`, and the specificity at that threshold in `spec94`. That is the price, in false alarms, of matching the brochure on your data.

*Try this on your own first, without an AI assistant (you can switch off Colab's automatic code suggestions under Tools > Settings > AI assistance). Use the hint if you get stuck, and open the solution only after a real attempt.*

*Hint:* Filter `sweep` to the rows with sensitivity >= 0.94, take the largest threshold with `.max()`, then read the specificity in that row with `.loc[...]` and `.iloc[0]`.

In [ ]:
# Your code here


print("Threshold for >= 94% sensitivity:", t94, " specificity there:", round(spec94, 3))

<details><summary>Show a solution</summary>

```python
ok = sweep[sweep["sensitivity"] >= 0.94]
t94 = ok["threshold"].max()
spec94 = ok.loc[ok["threshold"] == t94, "specificity"].iloc[0]

print("Threshold for >= 94% sensitivity:", t94, " specificity there:", round(spec94, 3))
```

</details>

In [ ]:
r = sens_spec(det["truth"], (det["score"] >= t94).astype(int))
assert r["sensitivity"] >= 0.94, "At t94 the sensitivity should be at least 0.94."
assert abs(r["specificity"] - spec94) < 0.01, "spec94 should be the specificity at t94."
higher = sweep[sweep["threshold"] > t94 + 1e-9]
assert (higher["sensitivity"] < 0.94).all(), "There is a higher threshold that still reaches 0.94."
print("Check passed")

---
## Part 4: Subgroups

An overall number can hide a group where the tool does much worse. **Subgroup analysis** repeats the evaluation inside each group: site, scanner model, age band, sex. Differences can come from different patients (sicker, younger), different equipment, or different ways of taking the image (portable bedside X-rays are harder to read). A tool that is worse for one group is a safety and fairness problem even when the average looks good.

**Mode: Reuse**

#### Task 10. Sensitivity at each site

Use your `sens_spec()` (just saved to your toolkit) on site A and site B separately.

In [ ]:
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import sens_spec

site_a = det[det["site"] == "A"]
site_b = ____
res_a = sens_spec(site_a["truth"], site_a["pred"])
res_b = ____
print(f"Site A: sensitivity {res_a['sensitivity']:.1%}, specificity {res_a['specificity']:.1%}")
print(f"Site B: sensitivity {res_b['sensitivity']:.1%}, specificity {res_b['specificity']:.1%}")

<details><summary>Show a solution</summary>

```python
import importlib
importlib.reload(toolkit)    # load the newest saved version of your toolkit
from toolkit import sens_spec

site_a = det[det["site"] == "A"]
site_b = det[det["site"] == "B"]
res_a = sens_spec(site_a["truth"], site_a["pred"])
res_b = sens_spec(site_b["truth"], site_b["pred"])
print(f"Site A: sensitivity {res_a['sensitivity']:.1%}, specificity {res_a['specificity']:.1%}")
print(f"Site B: sensitivity {res_b['sensitivity']:.1%}, specificity {res_b['specificity']:.1%}")
```

</details>

In [ ]:
assert rl_data.check(3, GITHUB_USERNAME, "sens_a", res_a["sensitivity"])
assert rl_data.check(3, GITHUB_USERNAME, "sens_b", res_b["sensitivity"]), "res_b should be sens_spec() on the site B rows."
print("Check passed")

**Mode: Direct the AI**

#### Task 11. A subgroup chart

Ask an AI assistant for a grouped bar chart comparing sensitivity and specificity at sites A and B, with the values written on the bars. Paste the code, run it, run the test, and fill in the log.

Suggested prompt for your AI assistant (adapt it as you like):

> I have two Python dicts, res_a and res_b, each with keys "sensitivity" and "specificity" (fractions 0-1),
> for site A and site B. Write matplotlib code for a grouped bar chart: sites on the x axis, one bar for sensitivity
> and one for specificity at each site, values written on top of the bars, y axis from 0 to 1.1, and a title.
> Store the axes in a variable called ax.

In [ ]:
# Paste the AI's code here

<details><summary>Show a reference solution (only after trying with the AI)</summary>

```python
sub = pd.DataFrame({
    "Sensitivity": [res_a["sensitivity"], res_b["sensitivity"]],
    "Specificity": [res_a["specificity"], res_b["specificity"]],
}, index=["Site A", "Site B"])

fig, ax = plt.subplots(figsize=(6, 4))
sub.plot(kind="bar", ax=ax, color=["#0D9488", "#F59E0B"], rot=0)
for container in ax.containers:
    ax.bar_label(container, fmt="%.2f")
ax.set_ylim(0, 1.1)
ax.set_ylabel("Rate")
ax.set_title("Pneumothorax detector by site (threshold 0.5)")
plt.show()
```

</details>

In [ ]:
assert len(ax.patches) >= 4, "The chart should have 4 bars: 2 metrics x 2 sites. Store the axes in a variable called ax."
assert ax.get_title() != "", "Ask for a title."
print("Check passed")

**AI-use log (Task 11).** Double-click this cell to edit it.

- Tool I used:
- My prompt (paste it):
- What I changed in the AI's code, and why:
- Did the test cell pass on the first try? If not, what was wrong?

### Putting it into words

Compare your results with the brochure. Is the overall sensitivity near 94%? Is it the same at both sites? A sentence for a report might read (with your own numbers, which will differ): "At a threshold of 0.5 the tool's sensitivity was 85% (site A 92%, site B 77%), below the vendor's reported 94%; reaching 94% required lowering the threshold to 0.40, which cut specificity from 93% to 78%." What would you recommend to the department before buying, and what else would you want to know about site B?

---
## Report back

You can now answer the question: what does the detector get on local data, and is it equal at both sites? Run the cell below. It prints the numbers to enter on the Lab 3 page on the AI2Practice site, together with your GitHub username (use the same one there). If the page says a number is off, it gives you a hint first.

In [ ]:
REPORT = {
    "sens": round(100 * m["sensitivity"], 1),
    "spec": round(100 * m["specificity"], 1),
    "sens_b": round(100 * res_b["sensitivity"], 1),
}
print("Report back for Lab 3")
print("GitHub username:", rl_data.normalize_username(GITHUB_USERNAME))
LABELS = {
    "sens": "Sensitivity at threshold 0.5 (%)",
    "spec": "Specificity at threshold 0.5 (%)",
    "sens_b": "Sensitivity at site B (%)",
}
for key, value in REPORT.items():
    print(f"  {LABELS[key]}: {value}")

---
## Save to GitHub

**1. Save the notebook.** In Colab, choose **File > Save a copy in GitHub**. The first time, Colab asks you to sign in to GitHub and allow access. Then set:

- Repository: `<your-username>/CA2i-Research-Launchpad` (your fork)
- Branch: `main`
- File path: `lesson-notebooks/Lab_3_Evaluating_an_AI_Tool.ipynb`
- Commit message: `Lab 3 finished`

Leave "Include a link to Colab" ticked and click **OK**. A **commit** is a saved snapshot of your files with a short message, so you can always see what changed and go back.

**2. Save your toolkit.** Colab forgets files like `toolkit.py` when the session ends, so keep your copy on GitHub too:

1. In the left sidebar of Colab, click the folder icon (Files). Find `toolkit.py`, click the three dots next to it, and choose **Download**.
2. On GitHub, open your fork (`github.com/<your-username>/CA2i-Research-Launchpad`) and click into the `lesson-notebooks` folder.
3. Click **Add file > Upload files**, drop in `toolkit.py`, write the message `Lab 3: add sens_spec()`, and click **Commit changes**. It replaces the starter copy.

Next time, the setup cell downloads `toolkit.py` from your fork, so `sens_spec()` is there waiting for you.

That's it for Lab 3. Go back to the lab page, enter your report-back numbers and write one or two sentences on what you found.